In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from utils import TFLiteEvalRecord

repo_root = Path.cwd().parent
results_dir = repo_root / "results"

records = []
for p in sorted(results_dir.glob("*.jsonl")):
    with p.open("r", encoding="utf-8") as f:
        lines = f.readlines()
    if not lines:
        continue
    last = json.loads(lines[-1])
    records.append(TFLiteEvalRecord.model_validate(last))

records = sorted(records, key=lambda r: r.model_name)
print(f"Loaded {len(records)} runs from {results_dir}")


2026-04-16 16:02:40.172379: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 16:02:40.182277: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 16:02:40.197564: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 16:02:40.197596: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 16:02:40.207633: I tensorflow/core/platform/cpu_feature_gua

Loaded 0 runs from /home/nathan/Documents/tiny-chirp-microflow/results


2026-04-16 16:02:41.858380: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-16 16:02:41.888348: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-16 16:02:41.889331: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [ ]:
plt.figure(figsize=(7, 4))
for rec in records:
    if rec.train.roc_fpr is None or rec.train.roc_tpr is None or rec.train.auc is None:
        continue
    plt.plot(
        rec.train.roc_fpr,
        rec.train.roc_tpr,
        label=f"{rec.model_name} (AUC={rec.train.auc:.2f})",
    )

plt.plot([0, 1], [0, 1], "k--", linewidth=0.8)
plt.xscale("log")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves")
plt.legend()
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
rows = []
for rec in records:
    m = rec.train
    rows.append(
        {
            "Model": rec.model_name,
            "Threshold (t)": m.threshold,
            "Acc.": m.accuracy,
            "Precision": m.precision,
            "Recall": m.recall,
            "F2": m.f2,
        }
    )

df_train = pd.DataFrame(rows).sort_values("Model")

df_train.style.format(
    {
        "Threshold (t)": "{:.2e}",
        "Acc.": "{:.2f}",
        "Precision": "{:.2f}",
        "Recall": "{:.2f}",
        "F2": "{:.2f}",
    }
)


In [ ]:
rows = []
for rec in records:
    m = rec.test
    rows.append(
        {
            "Model": rec.model_name,
            "Accuracy": m.accuracy,
            "Precision": m.precision,
            "Recall": m.recall,
            "F2": m.f2,
            "Avg Inference (ms)": (
                m.avg_inference_time_ms if m.avg_inference_time_ms is not None else np.nan
            ),
        }
    )

df_test = pd.DataFrame(rows).sort_values("Model")
df_test.style.format(
    {
        "Accuracy": "{:.2f}",
        "Precision": "{:.2f}",
        "Recall": "{:.2f}",
        "F2": "{:.2f}",
        "Avg Inference (ms)": "{:.3f}",
    }
)
